<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/pgt_II_pol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import PolynomialFeatures

# ══════════════════════════════════════════════════════════════════════════════
# PARAMÈTRES
# ══════════════════════════════════════════════════════════════════════════════
params = {
    "n_ambiant":     4,
    "deg_P":         3,
    "n_terms_poly":  20,
    "seeds": {1: 391, 2: 820215, 3: 541,
              42: 412156, 43: 941752, 44: 113380, 11: 44546, 12: 2154},
    "weights":       {0: 0, 1: 1, 2: 0.12, 3: 0.03},
    "n_train":       10,
    "n_test":        2000,
    "n_unlabeled":   4000,
    "n_functions":   200,      # nb de fonctions de base (taille dictionnaire)
    "embed_dim":     30,       # nb de vecteurs propres retenus par couche
    "lambda_G":      1,     # régularisation matrice de Gram
    "n_layers":      1,        # nb de couches de plongement
    "activation":    "none",   # "tanh" | "sigmoid" | "none"
    "k_proj":        7,        # dimension des projections aleatoires
    "n_proj":        1,       # nombre de projections par couche
    "n_deg":         8,        # degre max des polynomes
}

# ══════════════════════════════════════════════════════════════════════════════
# 1. POLYNÔMES ALÉATOIRES (s'annulent en 0, normalisés)
# ══════════════════════════════════════════════════════════════════════════════
def random_sparse_polynomial(d, degree, n_terms, seed=None):
    rng = np.random.default_rng(seed)
    all_indices = [exp for exp in product(range(degree+1), repeat=d)
                   if sum(exp) <= degree]
    indices = rng.choice(all_indices, size=n_terms, replace=False)
    coeffs  = rng.normal(size=n_terms)
    def P(x):
        y = np.zeros(x.shape[0])
        for c, alpha in zip(coeffs, indices):
            term = np.ones(x.shape[0])
            for j, e in enumerate(alpha):
                if e > 0: term *= x[:, j]**e
            y += c * term
        return y
    zero_val = sum(c for c, alpha in zip(coeffs, indices)
                   if all(e == 0 for e in alpha))
    def P_zero(x): return P(x) - zero_val
    return P_zero, indices, coeffs

def normalize_polynomial(P, indices, coeffs):
    max_c = np.max(np.abs(coeffs))
    if max_c > 0:
        nc = coeffs / max_c
        def Pn(x):
            y = np.zeros(x.shape[0])
            for c, alpha in zip(nc, indices):
                term = np.ones(x.shape[0])
                for j, e in enumerate(alpha):
                    if e > 0: term *= x[:, j]**e
                y += c * term
            return y
        return Pn
    return P

P1, i1, c1 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"],
                                        params["n_terms_poly"], params["seeds"][1])
P2, i2, c2 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"],
                                        params["n_terms_poly"], params["seeds"][2])
P3, i3, c3 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"],
                                        params["n_terms_poly"], params["seeds"][3])
P1 = normalize_polynomial(P1, i1, c1)
P2 = normalize_polynomial(P2, i2, c2)
P3 = normalize_polynomial(P3, i3, c3)

def Q(x): return P1(x) * P2(x) * P3(x)

# ══════════════════════════════════════════════════════════════════════════════
# 2. GÉNÉRATION DU CLOUD SUR {Q=0}, CENTRÉ EN 0
# ══════════════════════════════════════════════════════════════════════════════
def grad_Q(X):
    h = 1e-5; d = X.shape[1]
    p1=P1(X); p2=P2(X); p3=P3(X)
    grad = np.zeros_like(X)
    for k in range(d):
        Xp=X.copy(); Xp[:,k]+=h; Xm=X.copy(); Xm[:,k]-=h
        dp1=(P1(Xp)-P1(Xm))/(2*h)
        dp2=(P2(Xp)-P2(Xm))/(2*h)
        dp3=(P3(Xp)-P3(Xm))/(2*h)
        grad[:,k]=dp1*p2*p3+p1*dp2*p3+p1*p2*dp3
    return grad

def project_to_Q_zero(X_init, n_steps=150, lr=0.05, tol=1e-4):
    X = X_init.copy()
    for _ in range(n_steps):
        q=Q(X); gq=grad_Q(X); gq2=2*q[:,None]*gq
        norm=np.linalg.norm(gq2,axis=1,keepdims=True)+1e-12
        X=X-lr*gq2/norm; X=np.clip(X,0.,1.)
        if np.abs(Q(X)).max()<tol*0.1: break
    return X, np.abs(Q(X))<tol

def sample_on_Q_zero(n_target, d, seed=None):
    rng=np.random.default_rng(seed); collected=[]; n_col=0
    while n_col < n_target:
        n_batch=max((n_target-n_col)*4, 200)
        Xi=rng.uniform(0,1,(int(n_batch),d)); Xp,conv=project_to_Q_zero(Xi)
        good=Xp[conv]
        if len(good)>0: collected.append(good); n_col+=len(good)
        print(f"  collectés: {n_col}/{n_target} ({conv.mean():.1%})", end='\r')
    print()
    return np.vstack(collected)[:n_target]

print("Génération du cloud sur {Q=0}...")
d = params["n_ambiant"]
X_train     = sample_on_Q_zero(params["n_train"],   d, seed=params["seeds"][42])
X_test      = sample_on_Q_zero(params["n_test"],    d, seed=params["seeds"][43])
X_unlabeled = sample_on_Q_zero(params["n_unlabeled"],d,seed=params["seeds"][44])

# ── MODIFICATION 1 : centrage en 0 ───────────────────────────────────────────
X_all_raw = np.vstack([X_train, X_unlabeled])
center    = X_all_raw.mean(axis=0)
X_train     -= center
X_test      -= center
X_unlabeled -= center
X_all        = np.vstack([X_train, X_unlabeled])
N_all        = len(X_all)

print(f"  X_train:{X_train.shape}  mean≈{X_train.mean(axis=0).round(3)}")
print(f"  |Q|_max train={np.abs(Q(X_train+center)).max():.2e}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. FONCTION CIBLE
# ══════════════════════════════════════════════════════════════════════════════
P_target, it, ct = random_sparse_polynomial(params["n_ambiant"], 7,
                                             params["n_terms_poly"], seed=params["seeds"][11])
P_target = normalize_polynomial(P_target, it, ct)

def target_function(X):
    return P_target(1.5*X + center)   # on évalue dans les coordonnées originales

y_train = target_function(X_train)
y_test  = target_function(X_test)
print(f"  y_train : mean={y_train.mean():.3f}  std={y_train.std():.3f}")

# ══════════════════════════════════════════════════════════════════════════════
# 4. MATRICE DE GRAM SOBOLEV
#    Moyennée sur le cloud (MODIFICATION 2)
#    Vecteurs propres normalisés en norme Sobolev (MODIFICATION 3)
# ══════════════════════════════════════════════════════════════════════════════
def activate(X, mode):
    """Activation pour normaliser l embedding."""
    if mode == "tanh":     return np.tanh(X)
    elif mode == "sigmoid": return 1 / (1 + np.exp(-X))
    else:                  return X

def make_projections(d, k_proj, n_proj, rng):
    Pi = rng.standard_normal((n_proj, k_proj, d))
    norms = np.linalg.norm(Pi, axis=2, keepdims=True)
    return Pi / (norms + 1e-12)

def poly_features_projected(X, Pi, n_deg):
    from itertools import product as ip
    n = X.shape[0]; n_proj = Pi.shape[0]; k = Pi.shape[1]
    Z = np.einsum('pki,ni->pnk', Pi, X)
    monomes = [a for a in ip(range(n_deg+1), repeat=k) if sum(a) <= n_deg]
    n_mon = len(monomes)
    Phi = np.zeros((n, n_proj * n_mon))
    for p in range(n_proj):
        Zp = Z[p]
        for m, alpha in enumerate(monomes):
            col = np.ones(n)
            for j, e in enumerate(alpha):
                if e > 0: col *= Zp[:, j] ** e
            Phi[:, p * n_mon + m] = col
    return Phi, monomes

def poly_gram_projected(X_cloud, Pi, n_deg, weights, lambda_G):
    from itertools import product as ip
    n = X_cloud.shape[0]; n_proj = Pi.shape[0]; k = Pi.shape[1]
    monomes = [a for a in ip(range(n_deg+1), repeat=k) if sum(a) <= n_deg]
    n_mon = len(monomes); n_funcs = n_proj * n_mon
    Z = np.einsum('pki,ni->pnk', Pi, X_cloud)
    G = np.zeros((n_funcs, n_funcs))
    for p in range(n_proj):
        Zp = Z[p]; sl = slice(p * n_mon, (p+1) * n_mon)
        Phi_p = np.zeros((n, n_mon))
        for m, alpha in enumerate(monomes):
            col = np.ones(n)
            for j, e in enumerate(alpha):
                if e > 0: col *= Zp[:, j] ** e
            Phi_p[:, m] = col
        w0 = weights.get(0, 0.)
        if w0: G[sl, sl] += w0 * (Phi_p.T @ Phi_p) / n
        w1 = weights.get(1, 0.)
        if w1:
            GradZ = np.zeros((n, n_mon, k))
            for m, alpha in enumerate(monomes):
                for j in range(k):
                    if alpha[j] > 0:
                        col = np.ones(n) * alpha[j]
                        for l, e in enumerate(alpha):
                            e2 = e - 1 if l == j else e
                            if e2 > 0: col *= Zp[:, l] ** e2
                        GradZ[:, m, j] = col
            GradX = np.einsum('nmj,jd->nmd', GradZ, Pi[p])
            G[sl, sl] += w1 * np.einsum('nmi,nli->ml', GradX, GradX) / n
    G += lambda_G * np.eye(n_funcs)
    return G

def poly_embedding(X_input, X_cloud, weights, lambda_G,
                   n_proj, k_proj, n_deg, embed_dim, rng, n_ambiant):
    from itertools import product as ip
    d = X_input.shape[1]
    Pi = make_projections(d, k_proj, n_proj, rng)
    Phi_cloud, monomes = poly_features_projected(X_cloud, Pi, n_deg)
    n_mon = len(monomes); n_funcs = n_proj * n_mon
    G = poly_gram_projected(X_cloud, Pi, n_deg, weights, lambda_G)
    eigvals, eigvecs = np.linalg.eigh(G)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]; eigvecs = eigvecs[:, order]
    start = 1
    k = max(min(embed_dim + start, (eigvals > 1e-4).sum()) - start, 1)
    V = eigvecs[:, start:start+k]; L = eigvals[start:start+k]
    print(f"  VP jetées (0 à {start-1}) : {eigvals[:start].round(4)}")
    print(f"  Top {k} VP : {L[:min(k,30)].round(1)}")
    print(f"  Ratio variance : {L.sum()/eigvals[start:][eigvals[start:]>0].sum():.3f}")
    print(f"  Base : {n_funcs} ({n_proj} proj x {n_mon} monomes deg<={n_deg} sur R^{k_proj})")
    V_norm    = V / np.sqrt(L)[None, :]
    embed_cld = Phi_cloud @ V_norm
    col_means = embed_cld.mean(axis=0)
    Phi_in, _ = poly_features_projected(X_input, Pi, n_deg)
    embed_in  = Phi_in @ V_norm - col_means
    return embed_in, Pi, V_norm, col_means, L, monomes


rng = np.random.default_rng(seed=0)

# Résultats bruts (sans plongement) pour référence
print("\n── Résultats BRUTS (sans plongement) ──")
gamma_grid = np.logspace(-3, 2, 20)
param_grid = {'alpha':[1e-8,1e-5,1e-3], 'gamma':gamma_grid}
kr_raw = GridSearchCV(KernelRidge(kernel='rbf'), param_grid, cv=5,
                      scoring='neg_mean_squared_error', n_jobs=-1)
kr_raw.fit(X_train, y_train)
mse_rbf_raw = mean_squared_error(y_test, kr_raw.predict(X_test))

poly_raw = PolynomialFeatures(degree=3, include_bias=False)
ridge_raw = Ridge(alpha=1e-4)
ridge_raw.fit(poly_raw.fit_transform(X_train), y_train)
mse_ridge_raw = mean_squared_error(y_test, ridge_raw.predict(poly_raw.transform(X_test)))

print(f"  RBF   : {mse_rbf_raw:.6f}  (γ={kr_raw.best_params_['gamma']:.4f})")
print(f"  Ridge : {mse_ridge_raw:.6f}")

# ── Plongements itérés ────────────────────────────────────────────────────────
X_cur_train = X_train.copy()
X_cur_test  = X_test.copy()
X_cur_cloud = X_all.copy()

layer_results = []

for layer in range(params["n_layers"]):
    print(f"\n══ COUCHE {layer+1}/{params['n_layers']} "
          f"(dim entrée={X_cur_train.shape[1]}) ══")

    embed_train, Pi_l, V_norm_l, means_l, eigvals_l, monomes_l = poly_embedding(
        X_cur_train, X_cur_cloud, params["weights"], params["lambda_G"],
        params["n_proj"], params["k_proj"], params["n_deg"],
        params["embed_dim"], rng, params["n_ambiant"])

    def embed_new(X_in):
        Phi, _ = poly_features_projected(X_in, Pi_l, params["n_deg"])
        return Phi @ V_norm_l - means_l

    embed_test  = activate(embed_new(X_cur_test),  params["activation"])
    embed_train = activate(embed_train,             params["activation"])
    embed_cloud = activate(embed_new(X_cur_cloud),  params["activation"])

    print(f"  Embedding train : {embed_train.shape}  "
          f"range=[{embed_train.min():.3f}, {embed_train.max():.3f}]")

    # ── DIAGNOSTICS : structure des distances ─────────────────────────────────
    from scipy.stats import spearmanr
    rng_diag = np.random.default_rng(42)
    n_pairs  = 2000
    idx_i = rng_diag.integers(0, len(X_cur_train), n_pairs)
    idx_j = rng_diag.integers(0, len(X_cur_train), n_pairs)
    mask  = idx_i != idx_j
    idx_i, idx_j = idx_i[mask], idx_j[mask]

    dy      = np.abs(y_train[idx_i] - y_train[idx_j])
    eps_y   = np.percentile(dy, 25)
    delta_y = np.percentile(dy, 75)

    dx_before = np.linalg.norm(X_cur_train[idx_i] - X_cur_train[idx_j], axis=1)
    dx_after  = np.linalg.norm(embed_train[idx_i]  - embed_train[idx_j],  axis=1)

    dx_before_n = dx_before / (dx_before.mean() + 1e-12)
    dx_after_n  = dx_after  / (dx_after.mean()  + 1e-12)

    intra_b = dx_before_n[dy < eps_y].mean()
    inter_b = dx_before_n[dy > delta_y].mean()
    intra_a = dx_after_n[dy < eps_y].mean()
    inter_a = dx_after_n[dy > delta_y].mean()
    ratio_b = inter_b / (intra_b + 1e-12)
    ratio_a = inter_a / (intra_a + 1e-12)

    rho_b, _ = spearmanr(dx_before, dy)
    rho_a, _ = spearmanr(dx_after,  dy)

    tag1 = "AMELIORE"  if ratio_a > ratio_b else "DEGRADE"
    tag2 = "AMELIOREE" if rho_a   > rho_b   else "DEGRADEE"
    print(f"  DIAG 1 separation inter/intra : avant={ratio_b:.3f} apres={ratio_a:.3f} {tag1}")
    print(f"  DIAG 2 Spearman(dist,|dy|)    : avant={rho_b:.4f} apres={rho_a:.4f} {tag2}")

    # ── DIAG 3 : ratio dist_plongee / dist_euclidienne ───────────────────────
    # Pour chaque paire, on calcule dist_apres / dist_avant (normalisees).
    # Si le plongement compresse/redresse la variete, ce ratio doit etre
    # plus uniforme (variance plus faible) et/ou decroissant en moyenne
    # pour les paires proches sur la variete (petite dist_avant).
    eps_dx  = np.percentile(dx_before, 1)   # eviter division par zero
    valid   = dx_before > eps_dx
    ratio   = dx_after[valid] / dx_before[valid]
    # On normalise separement pour comparer les echelles
    dx_b_n  = dx_before[valid] / dx_before[valid].mean()
    dx_a_n  = dx_after[valid]  / dx_after[valid].mean()
    ratio_n = dx_a_n / dx_b_n

    # Paires proches vs eloignees dans l espace original
    q25 = np.percentile(dx_before[valid], 25)
    q75 = np.percentile(dx_before[valid], 75)
    ratio_proche   = ratio_n[dx_before[valid] < q25].mean()
    ratio_eloigne  = ratio_n[dx_before[valid] > q75].mean()
    cv_ratio       = ratio_n.std() / (ratio_n.mean() + 1e-12)  # coeff variation

    print(f"  DIAG 3 ratio dist_plongee/dist_euclid (normalise) :")
    print(f"    paires proches  (Q1 dist) : {ratio_proche:.4f}")
    print(f"    paires eloignees(Q3 dist) : {ratio_eloigne:.4f}")
    print(f"    coeff variation du ratio  : {cv_ratio:.4f}  "
          f"(plus petit = metrique plus isotrope)")
    rho_ratio, _ = spearmanr(dx_before[valid], ratio_n)
    print(f"    Spearman(dist_avant, ratio): {rho_ratio:.4f}  "
          f"(negatif = plongement compresse les paires proches)")

    # MODIFICATION 5 : RBF et Ridge après chaque couche
    kr_l = GridSearchCV(KernelRidge(kernel='rbf'), param_grid, cv=5,
                        scoring='neg_mean_squared_error', n_jobs=-1)
    kr_l.fit(embed_train, y_train)
    mse_rbf_l = mean_squared_error(y_test, kr_l.predict(embed_test))

    poly_l = PolynomialFeatures(degree=min(3, 8//max(1,embed_train.shape[1]//4)),
                                 include_bias=False)
    ridge_l = Ridge(alpha=1e-4)
    try:
        ridge_l.fit(poly_l.fit_transform(embed_train), y_train)
        mse_ridge_l = mean_squared_error(y_test,
                                          ridge_l.predict(poly_l.transform(embed_test)))
    except Exception:
        mse_ridge_l = float('nan')

    print(f"  RBF   après couche {layer+1} : {mse_rbf_l:.6f}  "
          f"(γ={kr_l.best_params_['gamma']:.4f})")
    print(f"  Ridge après couche {layer+1} : {mse_ridge_l:.6f}")

    layer_results.append({
        'layer': layer+1,
        'mse_rbf':   mse_rbf_l,
        'mse_ridge': mse_ridge_l,
        'embed_dim': embed_train.shape[1],
        'eigvals':   eigvals_l,
    })

    # Mise à jour pour la couche suivante
    X_cur_train = embed_train
    X_cur_test  = embed_test
    X_cur_cloud = embed_cloud

# ══════════════════════════════════════════════════════════════════════════════
# RÉSUMÉ
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("RÉSUMÉ")
print("="*70)
print(f"{'Source':<25} {'RBF':>12} {'Ridge':>12}")
print(f"{'Brut (dim=' + str(d) + ')':<25} {mse_rbf_raw:>12.6f} {mse_ridge_raw:>12.6f}")
for r in layer_results:
    label = f"Couche {r['layer']} (dim={r['embed_dim']})"
    print(f"{label:<25} {r['mse_rbf']:>12.6f} {r['mse_ridge']:>12.6f}")
print(f"\nweights={params['weights']}   "
      f"lambda_G={params['lambda_G']}")
print(f"n_functions={params['n_functions']}  embed_dim={params['embed_dim']}  "
      f"activation={params['activation']}  n_layers={params['n_layers']}")

Génération du cloud sur {Q=0}...
  collectés: 44/10 (22.0%)
  collectés: 2040/2000 (21.0%)
  collectés: 4020/4000 (21.5%)
  X_train:(10, 4)  mean≈[ 0.07   0.002 -0.097  0.01 ]
  |Q|_max train=9.30e-05
  y_train : mean=-0.103  std=0.269

── Résultats BRUTS (sans plongement) ──
  RBF   : 0.404464  (γ=0.0113)
  Ridge : 0.292095

══ COUCHE 1/1 (dim entrée=4) ══
